# 공간군 실습

**Space Group · 대칭성**

결정의 병진·회전 등 공간 대칭을 분류하는 체계.

소재 분야에서 이해하기: 결정 구조를 대칭에 따라 분류한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 대칭 연산을 직접 찾아봅니다

2차원 격자에 어떤 회전·거울 연산이 성립하는지 수치로 확인합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_lattice(kind, repeat=6):
    if kind == 'square':
        vectors = np.array([[1.0, 0.0], [0.0, 1.0]])
    elif kind == 'hexagonal':
        vectors = np.array([[1.0, 0.0], [0.5, np.sqrt(3) / 2]])
    else:  # 직사각
        vectors = np.array([[1.0, 0.0], [0.0, 1.6]])
    indices = np.arange(-repeat, repeat + 1)
    grid = np.array([[i, j] for i in indices for j in indices])
    return grid @ vectors

def is_symmetry(points, matrix, tolerance=1e-6):
    transformed = points @ matrix.T
    from scipy.spatial import cKDTree
    tree = cKDTree(points)
    inner = np.linalg.norm(points, axis=1) < np.linalg.norm(points, axis=1).max() * 0.6
    distance, _ = tree.query(transformed[inner])
    return bool(np.all(distance < tolerance))

def rotation(degrees):
    angle = np.radians(degrees)
    return np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])

MIRROR_X = np.array([[1.0, 0.0], [0.0, -1.0]])
print('대칭 검사기를 준비했습니다.')

In [ ]:
for kind in ('square', 'hexagonal', 'rectangular'):
    points = make_lattice(kind)
    found = [str(int(degrees)) + 'deg' for degrees in (60, 90, 120, 180)
             if is_symmetry(points, rotation(degrees))]
    mirror = '거울(x축) 있음' if is_symmetry(points, MIRROR_X) else '거울(x축) 없음'
    print('%-12s 회전 대칭 %-18s %s' % (kind, ','.join(found) or '없음', mirror))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for axis, kind in zip(axes, ('square', 'hexagonal', 'rectangular')):
    points = make_lattice(kind, repeat=3)
    axis.scatter(points[:, 0], points[:, 1], s=18)
    axis.set_aspect('equal'); axis.set_title(kind)
plt.tight_layout(); plt.show()
print('격자 모양이 허용하는 대칭이 다릅니다. 3차원에서 병진까지 포함해 분류한 것이 230개의 공간군입니다.')
print('실무에서는 pymatgen 의 SpacegroupAnalyzer 로 구조 파일에서 공간군을 판정합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#space-group)을 여세요.